<a href="https://colab.research.google.com/github/jinw00ch01/Dacon_RobotWorldModel_ActionVideo/blob/main/colab/train_cosmos_ac.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# S3: Cosmos-Predict2.5-2B 행동 조건 미세조정 (Colab A100/H100)

Ultra 가 만든 잠재 캐시와 변환 가중치를 Google Drive `MyDrive/dacon_wm` 에서 읽고, 체크포인트를 같은 곳에 저장한다.
끊기면 **모두 실행**을 다시 누르면 마지막 체크포인트부터 이어서 학습한다.

준비 (사람): Drive 에 `dacon_wm` 폴더가 Ultra 에서 동기화되어 있어야 한다. 런타임 유형은 GPU (A100 또는 H100).

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
SHARED = '/content/drive/MyDrive/dacon_wm'
assert os.path.isdir(SHARED), 'Drive 에 dacon_wm 폴더가 없습니다'
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [2]:
# 코드: 공개 저장소의 main (실행한 커밋은 아래에 출력되고 체크포인트 config 에도 남는다)
!rm -rf /content/repo && git clone -q https://github.com/jinw00ch01/Dacon_RobotWorldModel_ActionVideo /content/repo
!cd /content/repo && git log --oneline -1
!pip -q install diffusers==0.40.0 peft==0.21.2 transformers==5.18.0 accelerate safetensors einops av opencv-python-headless pandas pyarrow
# Colab 에 깔린 torchao 0.10 은 peft 0.21 이 LoRA 를 만들 때 거부한다. 여기서는 쓰지 않으므로 지운다.
!pip -q uninstall -y torchao

b6be43d (HEAD -> main, origin/main, origin/HEAD) STATUS: queue finished after restart; idm_v2 results
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 77.4 MB/s eta 0:00:00


In [ ]:
# 잠재 캐시를 로컬 디스크로 복사 (Drive 에서 직접 읽으면 느림)
!mkdir -p /content/latents && rsync -a --info=progress2 "$SHARED/latents_240x320/" /content/latents/
os.environ['WM_SHARED'] = SHARED
os.environ['WM_ACTION_STATS'] = f'{SHARED}/so100_action_statistics.json'

  1,229,898,443  47%   18.50MB/s    0:01:03 (xfr#29, to-chk=39/69)

In [ ]:
# 학습. A100 80GB/H100 은 batch 8, A100 40GB 는 batch 4 로 시작.
# 8,000 스텝(배치 8 기준 약 2.8 에폭). 2,000 스텝마다 adapter_00XXXX.pt 를 남겨 Ultra 에서 홀드아웃 채점한다.
import torch
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
BATCH = 8 if mem > 60 else 4
RUN = f'{SHARED}/cosmos_ac/runs/v2_full'
!cd /content/repo && python -m wmgen.train_cosmos_ac --latents /content/latents --out "$RUN" --batch $BATCH --steps 8000 --save-every 500 --keep-every 2000